### Initialize the Environment:
##### Virtual Environment Commands

| Command | Linux/Mac | GitBash |
| ------- | --------- | ------- |
| Create | `python3 -m venv venv` | `python -m venv venv` |
| Activate | `source venv/bin/activate` | `source venv/Scripts/activate` |
| Install | `pip install -r requirements.txt` | `pip install -r requirements.txt` |
| Deactivate | `deactivate` | `deactivate` |
##### Select the Kernel (This will be in the Requirements.txt eventually)

Using the venv (Python 3.13.2) located  in venv/bin/python)

### **Project Overview & Plan**
Capstone Project for Code:You Data Analysis track. This project analyzes Beer Recipes for frequency of uploads for various beer styles, while capturing preferences of strength, hopiness and batch size.    The goal of the project is to demonstrate a general knowledge of Python (Pandas, Numpy, MatLibPlot, Plotly), SQL(MySQL), Tableu, Cursor and ChatGPT.<br><br>
**Data Sources:**

The datasets used in this project are all related to online beer recipes. One dataset contains the different styles of beer as recognized by the Beer Judge Certification Program (BJCP.org).
- [Beersmith Recipes](https://beersmithrecipes.com/recent/) - scraped data that contains certain fields of 100% of the all grain beer recipes that have been uploaded by users.

- [Brewers Friend All-Grain Recipes](https://www.brewersfriend.com/homebrew-recipes/all-grain/) - scraped data that contains select fields of 100% of the all-grain beer recipes that have been uploaded by users.
- [BJCP - Judging Styles of Beer](https://github.com/ascholer/bjcp-styleview/blob/main/styles.json) - dataset that contains the criterea used to judge beer. Will help determine if recipes meet the criterea to be considered a specific style of beer.

In [63]:
import pandas as pd
# import numpy as np
import re # Imported "re" Python module to assist with parsing the data with pattern matching
# import matplotlib
from pandas import DataFrame
from typing import Optional # received warning about "float | None" when my type hint only included "float"
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FuncFormatter
# from rich.console import Console
# from rich.table import Table

Note: styles.json from: https://github.com/ascholer/bjcp-styleview

### **Import Data** and preview Data shapes

In [64]:
bs_df = pd.read_csv('beersmith_recipes.csv')
bf_df = pd.read_csv('bf_recipes.csv')
kag_df = pd.read_csv('recipeData.csv', encoding='ISO-8859-1')
styles_df = pd.read_json('styles.json')
print(bf_df.shape)
print(bs_df.shape)
print(kag_df.shape)
print(styles_df.shape)

(215580, 8)
(63121, 5)
(73861, 23)
(116, 28)


### Data Cleanup

#### **Beer Smith Recipes -** beersmith_recipes.csv = bs_df
Will analyze data, delete duplicates, remove what appears to be test data, resolve missing data, split the stats column into the individual components to match up with the Brewers friend data. While cleaning up the data, will create a function to clean up the data in the other datasets. In the Beer Smith data there were 31 rows missing the Recipe Name. Since the name wasn't crucial to the analysis of the recipes, but the recipe information was still of value, we assigned a generic name to each of the rows missing the Recipe Name.

In [65]:
print(bs_df.info())
print(bs_df.columns)
print(bf_df.info())
# print(kag_df.info())
# print(styles_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63121 entries, 0 to 63120
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Recipe Name  63090 non-null  object
 1   Recipe URL   63121 non-null  object
 2   Beer Style   63121 non-null  object
 3   Brewer       63121 non-null  object
 4   Stats        63121 non-null  object
dtypes: object(5)
memory usage: 2.4+ MB
None
Index(['Recipe Name', 'Recipe URL', 'Beer Style', 'Brewer', 'Stats'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215580 entries, 0 to 215579
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Title   215566 non-null  object 
 1   Style   215580 non-null  object 
 2   Size    215580 non-null  object 
 3   OG      215580 non-null  float64
 4   FG      215580 non-null  float64
 5   ABV     215580 non-null  float64
 6   IBU     215580 non-null  float64
 7   Color   215580 no

In [66]:
# Create a copy for cleaning the dataframe
bs_df_cleaned = bs_df.copy()

# Count null values for each column
null_counts = bs_df_cleaned.isnull().sum()

# Filter columns with null counts greater than zero
columns_with_nulls = null_counts[null_counts > 0].index.tolist()

print(null_counts)
print(columns_with_nulls)


Recipe Name    31
Recipe URL      0
Beer Style      0
Brewer          0
Stats           0
dtype: int64
['Recipe Name']


In [ ]:
# This function is to replace null values in the Recipe Name Column with a Generic
# unique name that begins with the first 2 characters of the dataframe name and is 
# incremented by 1 to keep the name unique. This was done because the data in the 
# rest of the columns contributed to the data and analysis.
# This code was written with a lot of back and forth with Perplexity.ai.

import inspect

def replace_column_nulls(df, target_column):
    # Replace nulls in one specific column with unique numbered values
    # Get dataframe variable name safely
    try:
        caller_frame = inspect.currentframe().f_back  # type: ignore
        df_name = [k for k, v in caller_frame.f_locals.items() if v is df][0]  # type: ignore
        prefix = df_name[:2].upper()
    except (AttributeError, IndexError):  # Catch only relevant errors
        prefix = "DF"

    # Create column-specific generator
    def col_generator():
        counter = 1
        while True:
            yield f"{prefix} No Name {counter}"
            counter += 1

    # Only process specified column
    mask = df[target_column].isnull()
    num_nulls = mask.sum()
    
    if num_nulls > 0:
        gen = col_generator()
        replacements = [next(gen) for _ in range(num_nulls)]
        df.loc[mask, target_column] = replacements
        
        # Show changes
        print(f"Replaced {num_nulls} nulls in {target_column}")
        print("Example replacement:", replacements[0])
    
    return df

In [ ]:
# Call the function
bf_df_cleaned = replace_column_nulls(bs_df_cleaned, 'Recipe Name')

Replaced 31 nulls in Recipe Name
Example replacement: BS No Name 1


In [69]:
def clean_bs_df(df) -> DataFrame :
    """
    Further Clean the Beer Smith DataFrame by deleting duplicates and eliminating 
    rows with no value in the 'Beer Style' column.

    Parameters:
   (pd.DataFrame): The dataframe to be cleaned

    Returns:
    pd.DataFrame: The cleaned Beer Smith DataFrame.
    """
    # Create a copy for cleaning the dataframe
    # bs_df_cleaned = bs_df.copy()

    # Delete all duplicates based on the value in the column "Recipe URL"
    df.drop_duplicates(subset=['Recipe URL'], keep='first', inplace=True, ignore_index=True)

    # Eliminate rows with no value in column "Beer Style". In Data Wrangler I noticed columns with only a pair of parentheses in the column. 
    df = df[df['Beer Style'] != '()']

    return df


In [70]:
# Clean the Beer Smith DataFrame
bs_df_cleaned = clean_bs_df(bs_df_cleaned)
print(bs_df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
Index: 56761 entries, 0 to 56767
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Recipe Name  56761 non-null  object
 1   Recipe URL   56761 non-null  object
 2   Beer Style   56761 non-null  object
 3   Brewer       56761 non-null  object
 4   Stats        56761 non-null  object
dtypes: object(5)
memory usage: 2.6+ MB
None


In [71]:
# Confirming deletion of duplicates
print(bs_df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
Index: 56761 entries, 0 to 56767
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Recipe Name  56761 non-null  object
 1   Recipe URL   56761 non-null  object
 2   Beer Style   56761 non-null  object
 3   Brewer       56761 non-null  object
 4   Stats        56761 non-null  object
dtypes: object(5)
memory usage: 2.6+ MB
None


In [72]:
bs_df_cleaned

,Recipe Name,Recipe URL,Beer Style,Brewer,Stats
0,Super Magnifico Mexican Lager 8g - Solo v1,https://beersmithrecipes.com/viewrecipe/513713...,Cream Ale ( 1C),cdburg,"OG: 1.044 (10.9° P), Bitterness: 15.2 IBUs, AB..."
1,Clemens Honey Stout - 12 gal,https://beersmithrecipes.com/viewrecipe/209010...,Imperial Stout (20C),stevclem,"OG: 1.097 (23.2° P), Bitterness: 74.4 IBUs, AB..."
2,Modelo Especial,https://beersmithrecipes.com/viewrecipe/494521...,Vienna Lager ( 7A),jonsl8,"OG: 1.045 (11.2° P), Bitterness: 14.4 IBUs, AB..."
3,GammaRay,https://beersmithrecipes.com/viewrecipe/223328...,New England IPA (21B),rickkickin,"OG: 1.060 (14.9° P), Bitterness: 64.5 IBUs, AB..."
4,Rockaway Chocolate Peanut Butter Stout 2 Batch 2,https://beersmithrecipes.com/viewrecipe/510436...,Sweet Stout (16A),HiawathaBrewing,"OG: 1.059 (14.5° P), Bitterness: 29.5 IBUs, AB..."
...,...,...,...,...,...
56763,Garbage Brown,https://beersmithrecipes.com/viewrecipe/212/ga...,American Brown Ale (10C),wyzazz,"OG: 1.053 (13.0° P), Bitterness: 27.1 IBUs, AB..."
56764,Mild (110),https://beersmithrecipes.com/viewrecipe/204/mi...,Mild (11A),bonjour,"OG: 1.031 (7.8° P), Bitterness: 19.2 IBUs, ABV..."
56765,Simcoe Mild (99),https://beersmithrecipes.com/viewrecipe/202/si...,Mild (11A),bonjour,"OG: 1.031 (7.8° P), Bitterness: 13.9 IBUs, ABV..."
56766,"Malted Bliss, Wedding Barley Wine",https://beersmithrecipes.com/viewrecipe/200/ma...,English Barleywine (19B),bonjour,"OG: 1.137 (31.5° P), Bitterness: 45.9 IBUs, AB..."


#### Splitting the Stats & Beer Style columns into individual columns of Data: 
Beer Style "Dark Mild (13A)"
Stats "OG: 1.073 (17.7° P), Bitterness: 34.5 IBUs, ABV: 6.9 %"

| Style|Style Number|
|:----------:|:----------:|
| Dark Mild| 13A| 

| OG| Plato| IBU| ABV|FG |
|:----------:|:----------:|:----------:|:----------:|:----------:|
| 1.073| 17.7| 34.5 | 6.9 |Calc|


In [73]:
def parse_columns(df: pd.DataFrame) -> pd.DataFrame:
    def extract_values(row: pd.Series) -> pd.Series:
        stats = row['Stats']
        beer_style = row['Beer Style']
        
        style_match = re.match(r'(.*?)\s*\((.*?)\)', beer_style)
        
        return pd.Series({
            'Style': style_match.group(1).strip() if style_match else None,
            'Style Number': style_match.group(2) if style_match else None,
            'OG': extract_float(stats, r'OG: (\d+\.\d+)'),
            'Plato': extract_float(stats, r'(\d+\.\d+)° P'),
            'IBU': extract_float(stats, r'Bitterness: (\d+\.\d+)'),
            'ABV': extract_float(stats, r'ABV: (\d+\.\d+)'),
        })

# was getting a warning about "float | None" and Perplexity.ai explained why and how to fix

    def extract_float(text: str, pattern: str) -> Optional[float]:
        match = re.search(pattern, text)
        return float(match.group(1)) if match else None

    new_columns = df.apply(extract_values, axis=1)
    df = pd.concat([df, new_columns], axis=1) # Add new extracted columns
    df = df.drop(columns=["Beer Style", "Stats", "Recipe URL", "Brewer"])  # Drop original columns
    

    return df

In [74]:
bs_df_cleaned = parse_columns(bs_df_cleaned)


In [75]:
bs_df_cleaned

,Recipe Name,Style,Style Number,OG,Plato,IBU,ABV
0,Super Magnifico Mexican Lager 8g - Solo v1,Cream Ale,1C,1.044,10.9,15.2,4.7
1,Clemens Honey Stout - 12 gal,Imperial Stout,20C,1.097,23.2,74.4,10.0
2,Modelo Especial,Vienna Lager,7A,1.045,11.2,14.4,4.5
3,GammaRay,New England IPA,21B,1.060,14.9,64.5,6.0
4,Rockaway Chocolate Peanut Butter Stout 2 Batch 2,Sweet Stout,16A,1.059,14.5,29.5,7.3
...,...,...,...,...,...,...,...
56763,Garbage Brown,American Brown Ale,10C,1.053,13.0,27.1,5.0
56764,Mild (110),Mild,11A,1.031,7.8,19.2,2.9
56765,Simcoe Mild (99),Mild,11A,1.031,7.8,13.9,2.9
56766,"Malted Bliss, Wedding Barley Wine",English Barleywine,19B,1.137,31.5,45.9,15.0


#### Balling Formula for Final Gravity (FG) Calculation (ChatGPT)
To estimate the **Final Gravity (FG)** using the **Original Gravity (OG) in Plato (°P)** and the **Alcohol by Volume (ABV)**, we will use the **Balling formula**:


-Step 1: Calculate Residual Extract (re) in degrees Plato
re = (plato / 1.25) - (abv / 0.79)

-Step 2: Convert re from degrees Plato to Specific Gravity (SG)
fg = 1 + (re / (258.6 - ((re / 258.2) * 227.1)))


In [77]:
def calculate_final_gravity(plato, abv):
    """
    Calculate Final Gravity (FG) using the Balling formula.

    Parameters:
    plato (float): Original Gravity in degrees Plato.
    abv (float): Alcohol by volume percentage.
    re (float): Residual Extract in degrees Plato.

    Returns:
    float: Estimated Final Gravity (FG).
    """
    # Step 1: Calculate Residual Extract (RE) in degrees Plato
    re = (plato / 1.25) - (abv / 0.79)

    # Step 2: Convert RE from degrees Plato to Specific Gravity (SG)
    fg = 1 + (re / (258.6 - ((re / 258.2) * 227.1)))

    return fg


def add_final_gravity_column(df):
    """
    Add a new column 'FG' to the DataFrame with calculated Final Gravity.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'Plato' and 'ABV' columns.

    Returns:
    None: Modifies the DataFrame in place by adding a new column 'FG'.
    """
    # Apply the calculate_final_gravity function on every row, creating a new column
    # and locating it after the Plato column to match the Brewers Friend data frame
    plato_index = df.columns.get_loc('Plato')
    df.insert(
        loc=plato_index + 1,
        column='FG',
        value=df.apply(lambda row: round(calculate_final_gravity(row['Plato'], row['ABV']), 3), axis=1)
)

    


In [78]:
add_final_gravity_column(bs_df_cleaned)
print(bs_df_cleaned)

                                            Recipe Name               Style  \
0            Super Magnifico Mexican Lager 8g - Solo v1           Cream Ale   
1                          Clemens Honey Stout - 12 gal      Imperial Stout   
2                                       Modelo Especial        Vienna Lager   
3                                              GammaRay     New England IPA   
4      Rockaway Chocolate Peanut Butter Stout 2 Batch 2         Sweet Stout   
...                                                 ...                 ...   
56763                                     Garbage Brown  American Brown Ale   
56764                                        Mild (110)                Mild   
56765                                  Simcoe Mild (99)                Mild   
56766                 Malted Bliss, Wedding Barley Wine  English Barleywine   
56767                           More LPANE Goodness APA   American Pale Ale   

      Style Number     OG  Plato     FG   IBU   ABV